In [1]:
import pandas as pd
from catboost import CatBoostClassifier

#### Load the train data and drop irrelevant features


In [2]:
df = pd.read_csv('train.csv',sep=',')
y = df["Survived"]
df = df.drop(columns=["PassengerId","Name","Survived","Cabin","Ticket"],axis=1)

- There are some missing values in the Age column, so I decided to fill them with the mean age of each Pclass group.


In [3]:
print(df.isna().sum())
df_invalid_age=df[df['Age'].isna()] ##create a df for the data with NaN age
print(df_invalid_age["Pclass"].value_counts())##checking which Pclass has the most NaN age

mean_ages = df.groupby("Pclass")["Age"].mean()

## Filling the Nan ages
df["Age"] = df.apply(
    lambda row: mean_ages[row["Pclass"]] if pd.isna(row["Age"]) else row["Age"], axis=1
)


Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64
Pclass
3    136
1     30
2     11
Name: count, dtype: int64


There are only 2 missing data points, so I decided to examine each individually. and then fill based on the mode of their Pclass

In [4]:
print(df[df["Embarked"].isna()]) #it seems that both missing data are Pclass = 1
print(df[df["Pclass"]==1]["Embarked"].value_counts()) ## checking the most common embarked value for first class
df["Embarked"] = df["Embarked"].fillna(df[df["Pclass"]==1]["Embarked"].mode()[0]) # fill with the Pclass==1 mode
X_train = df

     Pclass     Sex   Age  SibSp  Parch  Fare Embarked
61        1  female  38.0      0      0  80.0      NaN
829       1  female  62.0      0      0  80.0      NaN
Embarked
S    127
C     85
Q      2
Name: count, dtype: int64


### Load the test dataset and adjust it to match the shape of the train dataset.

In [5]:
df_test = pd.read_csv('test.csv',sep=',')
passenger_id = df_test["PassengerId"]
df_test = df_test.drop(['Name','Cabin','PassengerId',"Ticket"],axis=1)

### Applying the same changes


In [6]:
print(df_test.isna().sum())
df_test_invalid_age=df_test[df_test['Age'].isna()]
print(df_test_invalid_age["Pclass"].value_counts())

mean_ages = df_test.groupby("Pclass")["Age"].mean()

df_test["Age"] = df_test.apply(
    lambda row: mean_ages[row["Pclass"]] if pd.isna(row["Age"]) else row["Age"], axis=1
)

print(df_test[df_test["Embarked"].isna()])
print(df_test[df_test["Pclass"]==1]["Embarked"].value_counts())
df_test["Embarked"] = df_test["Embarked"].fillna(df_test[df_test["Pclass"]==1]["Embarked"].mode()[0])
X_test = df_test

Pclass       0
Sex          0
Age         86
SibSp        0
Parch        0
Fare         1
Embarked     0
dtype: int64
Pclass
3    72
1     9
2     5
Name: count, dtype: int64
Empty DataFrame
Columns: [Pclass, Sex, Age, SibSp, Parch, Fare, Embarked]
Index: []
Embarked
C    56
S    50
Q     1
Name: count, dtype: int64


Choose the Model, train and predict

In [7]:
model = CatBoostClassifier(iterations= 1000,learning_rate=0.1,cat_features=["Embarked","Sex"],task_type='GPU',loss_function="Logloss")
model.fit(X_train, y)
y_pred = model.predict(X_test)
submission = pd.DataFrame({
   'PassengerId': passenger_id,  
  'Survived': y_pred                 
})
submission.to_csv('submission.csv',index=False)

0:	learn: 0.6265653	total: 31.5ms	remaining: 31.4s
1:	learn: 0.5838398	total: 55.2ms	remaining: 27.5s
2:	learn: 0.5409099	total: 76.2ms	remaining: 25.3s
3:	learn: 0.5145971	total: 95.1ms	remaining: 23.7s
4:	learn: 0.4904797	total: 120ms	remaining: 23.8s
5:	learn: 0.4672750	total: 139ms	remaining: 23.1s
6:	learn: 0.4532428	total: 159ms	remaining: 22.5s
7:	learn: 0.4428860	total: 172ms	remaining: 21.3s
8:	learn: 0.4354983	total: 187ms	remaining: 20.6s
9:	learn: 0.4306763	total: 212ms	remaining: 21s
10:	learn: 0.4265179	total: 225ms	remaining: 20.2s
11:	learn: 0.4184607	total: 250ms	remaining: 20.5s
12:	learn: 0.4130505	total: 268ms	remaining: 20.4s
13:	learn: 0.4064081	total: 295ms	remaining: 20.7s
14:	learn: 0.4040818	total: 313ms	remaining: 20.5s
15:	learn: 0.3994320	total: 337ms	remaining: 20.7s
16:	learn: 0.3945231	total: 362ms	remaining: 21s
17:	learn: 0.3923525	total: 387ms	remaining: 21.1s
18:	learn: 0.3912324	total: 404ms	remaining: 20.9s
19:	learn: 0.3901821	total: 421ms	remaini